# 02 — Preprocessing

Clean, merge, re-index, split, and save data ready for modelling.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import pickle
from scipy import sparse

from src.data import load_raw, clean_ratings, clean_anime, reindex, merge_datasets, train_test_split
from src.features import build_sparse_interactions, build_indicator_features

PROCESSED = '../data/processed/'

## 1. Load Raw Data

In [2]:
ratings_raw, anime_raw = load_raw()
print(f'Raw ratings: {ratings_raw.shape}')
print(f'Raw anime:   {anime_raw.shape}')

Raw ratings: (7813737, 3)
Raw anime:   (12294, 7)


## 2. Clean Ratings

Drop `-1` (watched but unrated) rows — these carry no preference signal.

In [3]:
ratings = clean_ratings(ratings_raw)
dropped = len(ratings_raw) - len(ratings)
print(f'Dropped {dropped:,} unrated rows ({dropped/len(ratings_raw)*100:.1f}%)')
print(f'Remaining: {ratings.shape}')
ratings.head()

Dropped 1,476,496 unrated rows (18.9%)
Remaining: (6337241, 3)


,user_id,anime_id,rating
47,1,8074,10
81,1,11617,10
83,1,11757,10
101,1,15451,10
153,2,11771,10


## 3. Clean Anime

- Fill missing genre/type with `'Unknown'`
- Coerce episodes to numeric
- Fill missing community rating with median

In [4]:
anime = clean_anime(anime_raw)
print('Nulls after cleaning:')
print(anime.isnull().sum())
anime.head()

Nulls after cleaning:
anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


## 4. Merge

In [5]:
dataset = merge_datasets(ratings, anime)
print(f'Merged dataset: {dataset.shape}')
# Confirm no cartesian explosion
assert len(dataset) == len(ratings), 'Row count mismatch after merge!'
dataset.head()

Merged dataset: (6337241, 9)


,user_id,anime_id,rating_x,name,genre,type,episodes,rating_y,members
0,1,8074,10,Highschool of the Dead,"Action, Ecchi, Horror, Supernatural",TV,12.0,7.46,535892.0
1,1,11617,10,High School DxD,"Comedy, Demons, Ecchi, Harem, Romance, School",TV,12.0,7.70,398660.0
2,1,11757,10,Sword Art Online,"Action, Adventure, Fantasy, Game, Romance",TV,25.0,7.83,893100.0
3,1,15451,10,High School DxD New,"Action, Comedy, Demons, Ecchi, Harem, Romance,...",TV,12.0,7.87,266657.0
4,2,11771,10,Kuroko no Basket,"Comedy, School, Shounen, Sports",TV,25.0,8.46,338315.0


## 5. Re-index User and Item IDs

Sparse matrix construction requires contiguous 0-based integer indices.

In [6]:
dataset, user_map, item_map = reindex(dataset)

n_users = len(user_map)
n_items = len(item_map)
print(f'n_users: {n_users:,}')
print(f'n_items: {n_items:,}')
dataset[['user_id', 'user_idx', 'anime_id', 'item_idx', 'rating_x']].head()

n_users: 69,600
n_items: 9,927


,user_id,user_idx,anime_id,item_idx,rating_x
0,1,0,8074,0,10
1,1,0,11617,1,10
2,1,0,11757,2,10
3,1,0,15451,3,10
4,2,1,11771,4,10


## 6. Train / Test Split

In [7]:
train, test = train_test_split(dataset, train_ratio=0.75)
print(f'Train: {train.shape} | Test: {test.shape}')

Train: (4752930, 11) | Test: (1584311, 11)


## 7. Build Sparse Interaction Matrices

In [8]:
sparse_train = build_sparse_interactions(train, n_users, n_items)
sparse_test  = build_sparse_interactions(test,  n_users, n_items)

user_features, item_features = build_indicator_features(n_users, n_items)

print(f'sparse_train: {sparse_train.shape} | nnz: {sparse_train.nnz:,}')
print(f'sparse_test:  {sparse_test.shape}  | nnz: {sparse_test.nnz:,}')
print(f'user_features: {user_features.shape}')
print(f'item_features: {item_features.shape}')

sparse_train: (69600, 9927) | nnz: 4,752,930
sparse_test:  (69600, 9927)  | nnz: 1,584,311
user_features: (69600, 69600)
item_features: (9927, 9927)


## 8. Save Processed Artefacts

In [9]:
import os
os.makedirs(PROCESSED, exist_ok=True)

# Save sparse matrices
sparse.save_npz(PROCESSED + 'sparse_train.npz', sparse_train.tocsr())
sparse.save_npz(PROCESSED + 'sparse_test.npz',  sparse_test.tocsr())
sparse.save_npz(PROCESSED + 'user_features.npz', user_features.tocsr())
sparse.save_npz(PROCESSED + 'item_features.npz', item_features.tocsr())

# Save ID maps
with open(PROCESSED + 'user_map.pkl', 'wb') as f:
    pickle.dump(user_map, f)
with open(PROCESSED + 'item_map.pkl', 'wb') as f:
    pickle.dump(item_map, f)

# Save cleaned dataframes
train.to_csv(PROCESSED + 'train.csv', index=False)
test.to_csv(PROCESSED + 'test.csv', index=False)
anime.to_csv(PROCESSED + 'anime_clean.csv', index=False)

print('Saved:')
for f in sorted(os.listdir(PROCESSED)):
    size = os.path.getsize(PROCESSED + f) / 1024
    print(f'  {f:<30} {size:>8.1f} KB')

Saved:
  anime_catalogue.png                57.7 KB
  anime_clean.csv                   900.4 KB
  item_features.npz                  28.0 KB
  item_map.pkl                      213.2 KB
  rating_distribution.png            36.1 KB
  ratings_per_user.png               38.8 KB
  sparse_test.npz                  3159.6 KB
  sparse_train.npz                 9486.4 KB
  test.csv                       171949.5 KB
  top20_anime.png                    89.0 KB
  top20_genres.png                   64.9 KB
  train.csv                      514112.8 KB
  user_features.npz                 189.9 KB
  user_map.pkl                     1503.4 KB


## 9. Summary

| Step | Detail |
|---|---|
| Dropped unrated rows | 1,476,496 (-1 ratings) |
| Cleaned anime | Filled missing genre, type, episodes, rating |
| Re-indexed IDs | Contiguous 0-based user_idx and item_idx |
| Train/test split | 75% / 25% |
| Sparse matrices | COO → CSR format saved as .npz |
| Saved artefacts | sparse matrices, ID maps, CSVs |